<a href="https://colab.research.google.com/github/kavyaayyappan1998/DATA266---2738/blob/main/Assignment5/Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 0 - Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
!pip -q install datasets transformers peft accelerate sentencepiece evaluate rouge_score
!pip -q install --upgrade torchao

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.3 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from peft import LoraConfig, TaskType, get_peft_model

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


Step 1 - Data Preparation

In [3]:
ds = load_dataset("neil-code/dialogsum-test")
ds

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})

In [4]:
# Dialogue is the input and summary is the target
def process_sample(example):
    return {
        "input": example["dialogue"],
        "target": example["summary"]
    }

processed_ds = ds.map(process_sample)

for i in [0, 1]:
    print(f"Processed sample {i + 1}")
    print("Input:")
    print(processed_ds["train"][i]["input"])
    print("\nTarget:")
    print(processed_ds["train"][i]["target"])
    print("\n" + "=" * 70 + "\n")

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Processed sample 1
Input:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

Target:
Mr. Smith's getting a check-up, a

Step 2 - Baseline Inference

In [5]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
baseline_model = baseline_model.to(device)

print("Model loaded:", model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-small


In [6]:
# Pick two different test dialogues
sample_indices = [0, 3]

test_dialogues = [
    processed_ds["test"][i]["input"]
    for i in sample_indices
]

reference_summaries = [
    processed_ds["test"][i]["target"]
    for i in sample_indices
]


def generate_summary(model, dialogue):
    prompt = "Summarize the following dialogue:\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=4
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

baseline_outputs = []

for i, dialogue in enumerate(test_dialogues):
    output = generate_summary(baseline_model, dialogue)
    baseline_outputs.append(output)

    print(f"Dialogue {i + 1}:")
    print(dialogue)

    print("\nReference:")
    print(reference_summaries[i])

    print("\nBaseline:")
    print(output)

    print("\n" + "=" * 70 + "\n")

Dialogue 1:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with 

In [7]:
baseline_df = pd.DataFrame({
    "Dialogue": test_dialogues,
    "Reference": reference_summaries,
    "Baseline": baseline_outputs
})

baseline_df.to_csv("baseline_outputs.csv", index=False)
baseline_df

,Dialogue,Reference,Baseline
0,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson helps #Person1# to write a memo to ...,"#Person2#: Thank you, sir."
1,#Person1#: You're finally here! What took so l...,#Person2# arrives late because of traffic jam....,Taking the subway would be a lot less stressfu...


Step 3 - Apply LoRA

In [8]:
# Fresh model for r=4
model_r4 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config_r4 = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model_r4 = get_peft_model(
    model_r4,
    lora_config_r4
)

model_r4.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


Step 4 - Fine-Tuning

In [9]:
# Use a small subset
train_size = min(
    500,
    len(processed_ds["train"])
)

val_size = min(
    100,
    len(processed_ds["validation"])
)

train_data = (
    processed_ds["train"]
    .shuffle(seed=42)
    .select(range(train_size))
)

val_data = (
    processed_ds["validation"]
    .shuffle(seed=42)
    .select(range(val_size))
)

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))

Training samples: 500
Validation samples: 100


In [10]:
def tokenize_batch(batch):
    prompts = [
        "Summarize the following dialogue:\n" + text
        for text in batch["dialogue"]
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_val = val_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_data.column_names
)

data_collator_r4 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r4
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [11]:
training_args_r4 = Seq2SeqTrainingArguments(
    output_dir="./results_r4",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-3,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

trainer_r4 = Seq2SeqTrainer(
    model=model_r4,
    args=training_args_r4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator_r4
)

train_result_r4 = trainer_r4.train()

Step,Training Loss
20,1.936254
40,1.547206
60,1.621865
80,1.740901
100,1.508124
120,1.596465
140,1.573141
160,1.537900
180,1.508455
200,1.450657


In [12]:
print(
    "Final training loss:",
    round(train_result_r4.training_loss, 4)
)

log_df_r4 = pd.DataFrame(
    trainer_r4.state.log_history
)

log_df_r4.tail(10)

Final training loss: 1.5512


,loss,grad_norm,learning_rate,epoch,step,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
9,1.450657,1.086522,0.000469,1.60,200,NaN,NaN,NaN,NaN,NaN
10,1.575673,0.679823,0.000416,1.76,220,NaN,NaN,NaN,NaN,NaN
11,1.516412,1.021767,0.000363,1.92,240,NaN,NaN,NaN,NaN,NaN
12,1.577251,0.752798,0.000309,2.08,260,NaN,NaN,NaN,NaN,NaN
13,1.412182,0.789169,0.000256,2.24,280,NaN,NaN,NaN,NaN,NaN
14,1.412010,0.960467,0.000203,2.40,300,NaN,NaN,NaN,NaN,NaN
15,1.425749,1.178120,0.000149,2.56,320,NaN,NaN,NaN,NaN,NaN
16,1.484978,1.176945,0.000096,2.72,340,NaN,NaN,NaN,NaN,NaN
17,1.520712,0.884521,0.000043,2.88,360,NaN,NaN,NaN,NaN,NaN
18,NaN,NaN,NaN,3.00,375,76.1035,19.71,4.928,1.858583e+14,1.551186


In [13]:
# Save r=4 adapter
r4_path = "./dialogsum_lora_r4"

model_r4.save_pretrained(r4_path)
tokenizer.save_pretrained(r4_path)

print("Saved to:", r4_path)

Saved to: ./dialogsum_lora_r4


Step 5 - Post-Fine-Tuning Inference

In [14]:
model_r4 = model_r4.to(device)

r4_outputs = []

for i, dialogue in enumerate(test_dialogues):
    output = generate_summary(
        model_r4,
        dialogue
    )

    r4_outputs.append(output)

    print(f"Dialogue {i + 1}:")
    print(dialogue)

    print("\nReference:")
    print(reference_summaries[i])

    print("\nAfter fine-tuning r=4:")
    print(output)

    print("\n" + "=" * 70 + "\n")

Dialogue 1:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with 

Step 6 - Output Comparison

In [15]:
comparison_r4 = pd.DataFrame({
    "Sample": [1, 2],
    "Reference": reference_summaries,
    "Before Fine-Tuning": baseline_outputs,
    "After Fine-Tuning r=4": r4_outputs
})

pd.set_option("display.max_colwidth", None)
comparison_r4

,Sample,Reference,Before Fine-Tuning,After Fine-Tuning r=4
0,1,Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.,"#Person2#: Thank you, sir.",#Person1# asks #Person2# to take a dictation for Ms. Dawson. #Person2# asks #Person1# to take a dictation for Ms. Dawson. #Person1# asks #Person2# to take a dictation for Ms. Dawson.
1,2,#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.,Taking the subway would be a lot less stressful than driving.,#Person2# gets stuck in traffic again. #Person1# thinks it's better if #Person2# started taking public transport system to work. #Person2# thinks it's better for the environment.


What changed?

Before fine-tuning, the model didn't summarize. For dialogue 1 it copied a dialogue line ("#Person2#: Thank you, sir."), and for dialogue 2 it gave one generic sentence. After LoRA fine-tuning, the model writes third-person summaries in the DialogSum style using #Person1#/#Person2#, and dialogue 2 captures the main points (traffic, public transport, environment). ROUGE-L improved from 0.03 to 0.27.

However, dialogue 1 is still weak: the model repeats the same sentence and gets the roles wrong, and it misses the memo about Instant Messaging.

Step 7 - Experiment with LoRA Rank r=16

In [16]:
# Fresh model for r=16
model_r16 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config_r16 = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model_r16 = get_peft_model(
    model_r16,
    lora_config_r16
)

model_r16.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [17]:
data_collator_r16 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r16
)

training_args_r16 = Seq2SeqTrainingArguments(
    output_dir="./results_r16",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-3,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

trainer_r16 = Seq2SeqTrainer(
    model=model_r16,
    args=training_args_r16,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator_r16
)

train_result_r16 = trainer_r16.train()

Step,Training Loss
20,1.937049
40,1.548194
60,1.620488
80,1.741024
100,1.508870
120,1.598965
140,1.571343
160,1.533590
180,1.499638
200,1.444341


In [18]:
print(
    "Final training loss:",
    round(train_result_r16.training_loss, 4)
)

log_df_r16 = pd.DataFrame(
    trainer_r16.state.log_history
)

log_df_r16.tail(10)

Final training loss: 1.5463


,loss,grad_norm,learning_rate,epoch,step,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
9,1.444341,0.534411,0.000469,1.60,200,NaN,NaN,NaN,NaN,NaN
10,1.567487,0.362938,0.000416,1.76,220,NaN,NaN,NaN,NaN,NaN
11,1.504512,0.502345,0.000363,1.92,240,NaN,NaN,NaN,NaN,NaN
12,1.572913,0.402482,0.000309,2.08,260,NaN,NaN,NaN,NaN,NaN
13,1.400819,0.421396,0.000256,2.24,280,NaN,NaN,NaN,NaN,NaN
14,1.405867,0.539088,0.000203,2.40,300,NaN,NaN,NaN,NaN,NaN
15,1.415131,0.645407,0.000149,2.56,320,NaN,NaN,NaN,NaN,NaN
16,1.482769,0.593804,0.000096,2.72,340,NaN,NaN,NaN,NaN,NaN
17,1.509811,0.465192,0.000043,2.88,360,NaN,NaN,NaN,NaN,NaN
18,NaN,NaN,NaN,3.00,375,55.2368,27.156,6.789,1.874390e+14,1.546299


In [19]:
# Save r=16 adapter
r16_path = "./dialogsum_lora_r16"

model_r16.save_pretrained(r16_path)
tokenizer.save_pretrained(r16_path)

print("Saved to:", r16_path)

Saved to: ./dialogsum_lora_r16


In [20]:
model_r16 = model_r16.to(device)

r16_outputs = []

for i, dialogue in enumerate(test_dialogues):
    output = generate_summary(
        model_r16,
        dialogue
    )

    r16_outputs.append(output)

    print(f"Dialogue {i + 1}:")
    print(dialogue)

    print("\nReference:")
    print(reference_summaries[i])

    print("\nr=4:")
    print(r4_outputs[i])

    print("\nr=16:")
    print(output)

    print("\n" + "=" * 70 + "\n")

Dialogue 1:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with 

Output Quality Check

In [21]:
rouge = evaluate.load("rouge")

outputs = {
    "Baseline": baseline_outputs,
    "LoRA r=4": r4_outputs,
    "LoRA r=16": r16_outputs
}

scores = {
    name: rouge.compute(
        predictions=preds,
        references=reference_summaries
    )
    for name, preds in outputs.items()
}

pd.DataFrame(scores).T[
    ["rouge1", "rouge2", "rougeL"]
].round(4)

,rouge1,rouge2,rougeL
Baseline,0.0303,0.0000,0.0303
LoRA r=4,0.2828,0.0572,0.2653
LoRA r=16,0.2828,0.0572,0.2653


Experiment Result:

r=16 has 4× more trainable parameters (688K vs 172K), but it gave the same results as r=4. Training loss was nearly equal (1.546 vs 1.551), and both models produced identical summaries for both dialogues, so the ROUGE scores are the same (ROUGE-L 0.265). Both ranks also show the same repetition problem on dialogue 1.

For a small model, a 500-sample subset, and 3 epochs, increasing the rank did not improve output quality, so r=4 is the more efficient choice here. A larger rank would likely matter more with more data or a bigger model.